# 06 · Joint fine-tuning: reviews in two languages, without forgetting

Notebook 05 compared full fine-tuning and LoRA on one language at a time and found that the learning rates that maximise the review score erase most of what SFT taught (chat and code). This notebook trains **one model per size** that rates reviews in Turkish *and* English and keeps its SFT abilities. These models are the starting point for RL.

Three changes from notebook 05:

1. **One mixed dataset:** Turkish reviews, English reviews, and a replay of the SFT data (one third of the windows).
2. **Loss averaged per window** (`loss_per_window=True`): a chat window has ~600 learned tokens and a review window 10-30, so with the token mean the replay would be ~95% of the loss. With the window mean each source counts by its share of windows.
3. **Selection includes forgetting**, with thresholds fixed here before any run (see section 9).

| Step | Runs | Time on A100 (estimate) |
|---|---:|---:|
| Baselines (SFT models) | 0 | ~5 min |
| Full fine-tuning, LR 3e-5 and 1e-4, both sizes | 4 | ~35 min |
| Test of the two selected models, samples, upload | 0 | ~10 min |

**Runtime:** *Select Kernel → Colab → New Colab Server → **A100***.

## 1. GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Free the GPU

Must run before anything touches CUDA in this kernel.

In [ ]:
# In VS Code + Colab, "Restart Kernel" can leave the old kernel (and its compile workers)
# running and holding GPU memory, which later causes CUDA out-of-memory errors.
import os
import signal

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation

me = os.getpid()
ps = subprocess.run(["ps", "-eo", "pid,args"], capture_output=True, text=True).stdout.splitlines()[1:]
stale = []
for line in ps:
    pid_text, _, args = line.strip().partition(" ")
    pid = int(pid_text)
    if pid == me:
        continue
    if "colab_kernel_launcher" in args or ("compile_worker" in args and f"--parent={me}" not in args):
        stale.append((pid, args[:80]))
for pid, args in stale:
    os.kill(pid, signal.SIGKILL)
    print("killed stale process", pid, args)
print("stale processes:", len(stale))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 3. Get the code

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 4. Run all tests

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout[-2500:], result.stderr[-1000:])
assert result.returncode == 0, "tests failed - fix before training"

## 5. Log in to Hugging Face

The token is typed into a hidden prompt and lives only in this session's memory.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

DATA_REPO = "Tbt-AI/glm-ladder-data"
RUNS_REPO = "Tbt-AI/glm-ladder-runs"

token = getpass("Hugging Face write token: ")
api = HfApi(token=token)
print("logged in as:", api.whoami()["name"])
for repo_id, repo_type in ((DATA_REPO, "dataset"), (RUNS_REPO, "model")):
    assert api.repo_info(repo_id, repo_type=repo_type).private, f"{repo_id} must be private"
    print("private:", repo_id)

## 6. Download and verify the data

Review data (both languages) and the SFT windows (train for replay, val for forgetting), about 1.3 GB. Every file's SHA-256 is checked against its manifest.

In [ ]:
import json
import time

from huggingface_hub import snapshot_download

from glm_ladder.checkpoint import sha256_file

DATA_DIR = Path("/content/data")
started = time.perf_counter()
snapshot_download(repo_id=DATA_REPO, repo_type="dataset", local_dir=DATA_DIR, token=token,
                  allow_patterns=["finetune/*", "sft/*"])
print(f"downloaded in {time.perf_counter() - started:.0f} s")

tokenizer_receipt = json.loads((REPO_DIR / "artifacts" / "receipts" / "tokenizer-16k.json").read_text())
ft_manifest = json.loads((DATA_DIR / "finetune" / "manifest.json").read_text())
sft_manifest = json.loads((DATA_DIR / "sft" / "manifest.json").read_text())
for manifest in (ft_manifest, sft_manifest):
    assert manifest["tokenizer_sha256"] == tokenizer_receipt["sha256"], "tokenizer mismatch"
for language, splits in ft_manifest["languages"].items():
    for split, info in splits.items():
        for file, digest in info["sha256"].items():
            assert sha256_file(DATA_DIR / "finetune" / language / file) == digest, f"{language}/{file}"
for source, splits in sft_manifest["sources"].items():
    for split, info in splits.items():
        for file, digest in info["sha256"].items():
            assert sha256_file(DATA_DIR / "sft" / source / split / file) == digest, f"sft {source}/{split}/{file}"
print(f"all files verified in {time.perf_counter() - started:.0f} s")

## 7. The mix

Review windows are used once each (one epoch). SFT replay adds half as many windows again, so it is one third of all windows, split between English and Turkish SFT in their natural 90 / 10 ratio. The loader draws every row of a batch from one source with these probabilities.

In [ ]:
import torch
from safetensors.torch import load_model

from glm_ladder import GLMLadderModel, ModelConfig
from glm_ladder.classify import LANGUAGES, evaluate_reviews
from glm_ladder.code_eval import load_tasks
from glm_ladder.runs import RunStore
from glm_ladder.sft_data import SFTSource
from glm_ladder.tokenizer import load_tokenizer

SEQ_LEN = ft_manifest["seq_len"]
REPLAY_FRACTION = 1 / 3
train = {lang: SFTSource(DATA_DIR / "finetune" / lang / "train") for lang in LANGUAGES}
val = {lang: SFTSource(DATA_DIR / "finetune" / lang / "val") for lang in LANGUAGES}
for name in sft_manifest["sources"]:
    train[f"sft_{name}"] = SFTSource(DATA_DIR / "sft" / name / "train")
    val[f"sft_{name}"] = SFTSource(DATA_DIR / "sft" / name / "val")

review_windows = sum(train[lang].num_windows for lang in LANGUAGES)
replay_windows = review_windows * REPLAY_FRACTION / (1 - REPLAY_FRACTION)
sft_total = sum(train[f"sft_{name}"].num_windows for name in sft_manifest["sources"])
windows = {lang: train[lang].num_windows for lang in LANGUAGES}
windows.update({f"sft_{name}": replay_windows * train[f"sft_{name}"].num_windows / sft_total
                for name in sft_manifest["sources"]})
EPOCH_WINDOWS = round(sum(windows.values()))
weights = {name: count / EPOCH_WINDOWS for name, count in windows.items()}
for name, weight in weights.items():
    print(f"{name:12s} weight {weight:.3f}  ~{windows[name]:,.0f} windows per epoch")
print(f"{EPOCH_WINDOWS:,} windows = {EPOCH_WINDOWS * SEQ_LEN / 1e6:.1f}M tokens per epoch")


def read_jsonl(path):
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]
    return [row["text"] for row in rows], [row["stars"] for row in rows]


reviews = {(lang, split): read_jsonl(DATA_DIR / "finetune" / lang / f"{split}.jsonl")
           for lang in LANGUAGES for split in ("val", "test")}
tokenizer = load_tokenizer(REPO_DIR / "tokenizer" / "glm-ladder-16k.json")
code_tasks = load_tasks(REPO_DIR / "evals" / "code_tasks.jsonl")
device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

BASES = {"s50m": "sft-s50m-001", "m100m": "sft-m100m-001"}
base_weights, base_states = {}, {}
for size, run in BASES.items():
    checkpoint = RunStore(api, RUNS_REPO, run).latest()
    def download(name):
        return Path(api.hf_hub_download(repo_id=RUNS_REPO, filename=f"runs/{run}/{checkpoint}/{name}", repo_type="model"))
    state = json.loads(download("trainer.json").read_text())
    weights_file = download("model.safetensors")
    assert sha256_file(weights_file) == state["model_sha256"], f"{run}: weights do not match their SHA-256"
    base_weights[size], base_states[size] = weights_file, {**state, "checkpoint": f"{RUNS_REPO}/runs/{run}/{checkpoint}"}
    print(f"{size}: {run} @ {checkpoint}")

## 8. Baselines: the SFT models

What fine-tuning must keep: the SFT validation losses (128 windows per language, as in every earlier evaluation) and the code reference loss on the 40 coding tasks in chat mode.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "scripts"))
from run_code_eval import evaluate as code_evaluate   # scripts/run_code_eval.py

from glm_ladder.evaluation import evaluate as loss_evaluate

SFT_VAL = [name for name in val if name.startswith("sft_")]


def load_base(size):
    model = GLMLadderModel(ModelConfig.from_dict(base_states[size]["model_config"]))
    load_model(model, str(base_weights[size]), strict=True)
    return model.to(device).eval()


def review_metrics(model, split):
    return {lang: evaluate_reviews(model, tokenizer, *reviews[(lang, split)], lang, batch_size=64, device=device)
            for lang in LANGUAGES}


def code_reference_loss(model):
    return code_evaluate(model, tokenizer, code_tasks, mode="chat", loss_only=True, log=lambda _: None)[1]["reference_loss"]


def keep_metrics(model):
    return {**loss_evaluate(model, {name: val[name] for name in SFT_VAL}, windows=128, batch_size=16, device=device),
            "code": code_reference_loss(model)}


def show(name, reviews_result, keep):
    print(f"{name:26s} " + "  ".join(f"{lang}: F1 {m['macro_f1']:.3f} acc {m['accuracy']:.3f}" for lang, m in reviews_result.items())
          + "  | " + "  ".join(f"{k} {v:.3f}" for k, v in keep.items()))


baseline = {}
for size in BASES:
    model = load_base(size)
    baseline[size] = {"val": review_metrics(model, "val"), "keep": keep_metrics(model)}
    show(f"SFT {size} (val)", baseline[size]["val"], baseline[size]["keep"])
    del model
    torch.cuda.empty_cache()

## 9. Training and the selection rule

Full fine-tuning (notebook 05: better than LoRA at every matched level of forgetting), one epoch of the mix, batch 128 windows, warmup 5%, cosine decay to 10%, no weight decay, `torch.compile` on.

**Selection rule, fixed before the runs.** For each size, among the runs whose forgetting stays within

- SFT validation loss: at most **+10%** in each language, and
- code reference loss: at most **+25%**,

pick the highest **mean validation macro-F1 over Turkish and English**. If no run is within both limits, pick the run with the smallest worst-case relative increase, and say so.

In [ ]:
import math
import shutil
from dataclasses import asdict

from glm_ladder.pretrain import PretrainConfig, run_pretraining
from glm_ladder.runs import MetricsLog

GPU = torch.cuda.get_device_name()
MICRO_BATCH = 32 if "A100" in GPU else 16
BATCH_ROWS = 128
WORKDIR = Path("/content/mix-runs")
OUT = Path("/content/finetune-mix")
OUT.mkdir(exist_ok=True)
LIMITS = {"sft": 0.10, "code": 0.25}
results = {}


def run_name(size, lr):
    return f"mix-{size}-lr{lr:.0e}"


def forgetting(size, keep):
    base = baseline[size]["keep"]
    return {name: keep[name] / base[name] - 1 for name in keep}


def within_limits(change):
    return all(change[name] <= LIMITS["sft"] for name in SFT_VAL) and change["code"] <= LIMITS["code"]


def finetune(size, lr):
    name = run_name(size, lr)
    if (OUT / f"{name}.json").exists():
        results[name] = json.loads((OUT / f"{name}.json").read_text())
        return results[name]
    steps = math.ceil(EPOCH_WINDOWS / BATCH_ROWS)
    cfg = PretrainConfig(run_name=name, total_tokens=EPOCH_WINDOWS * SEQ_LEN, seq_len=SEQ_LEN, batch_rows=BATCH_ROWS,
                         micro_batch=MICRO_BATCH, max_lr=lr, warmup_steps=max(5, steps // 20), weight_decay=0.0,
                         eval_every=max(1, steps // 4), eval_windows=128, log_every=25, checkpoint_minutes=1e9,
                         compile=True, loss_per_window=True)
    if (WORKDIR / name).exists():
        shutil.rmtree(WORKDIR / name)
    torch._dynamo.reset()                                     # drop the previous model's compiled graphs
    result = run_pretraining(cfg, ModelConfig.from_dict(base_states[size]["model_config"]),
                             train_sources=train, val_sources=val, weights=weights, store=None,
                             workdir=WORKDIR, device=device, init_from=base_weights[size],
                             metadata={"commit": commit, "stage": "finetune-mix", "init_from": base_states[size]["checkpoint"]})
    model = result["model"].eval()
    rows = MetricsLog(WORKDIR / name / "metrics.jsonl").rows()
    keep = keep_metrics(model)
    receipt = {
        "schema_version": "1.0", "kind": "finetune_mix_run", "run_name": name, "commit": commit, "gpu": GPU,
        "size": size, "max_lr": lr, "init_from": base_states[size]["checkpoint"], "weights": weights,
        "steps": result["step"], "tokens_seen": result["tokens_seen"],
        "train_seconds": round(sum(row["seconds"] for row in rows if "seconds" in row), 1),
        "final_eval": [row for row in rows if "eval" in row][-1]["eval"],
        "val": review_metrics(model, "val"), "keep": keep, "forgetting": forgetting(size, keep),
        "within_limits": within_limits(forgetting(size, keep)), "config": asdict(cfg),
    }
    (OUT / f"{name}.json").write_text(json.dumps(receipt, indent=2) + "\n")
    results[name] = receipt
    show(name + " (val)", receipt["val"], keep)
    print(f"{'':26s} forgetting " + "  ".join(f"{k} {v:+.1%}" for k, v in receipt["forgetting"].items())
          + f"  -> {'within' if receipt['within_limits'] else 'OUTSIDE'} the limits")
    del model, result
    torch.cuda.empty_cache()
    return receipt

## 10. Runs and selection

Two learning rates per size: 1e-4 (the rate that kept coding best in notebook 05) and 3e-5 (more conservative).

In [ ]:
LRS = (3e-5, 1e-4)
selected = {}
for size in BASES:
    runs = [finetune(size, lr) for lr in LRS]
    passing = [r for r in runs if r["within_limits"]]
    if passing:
        best = max(passing, key=lambda r: sum(r["val"][lang]["macro_f1"] for lang in LANGUAGES) / len(LANGUAGES))
        reason = "best mean val macro-F1 within the limits"
    else:
        best = min(runs, key=lambda r: max(r["forgetting"].values()))
        reason = "no run within the limits: smallest worst-case forgetting"
    selected[size] = best["run_name"]
    print(f"==> {size}: {best['run_name']} ({reason})")

## 11. Test

The two selected models and their SFT starting points on both test sets. For comparison, the single-language specialists of notebook 05 reached 0.572 / 0.586 (`s-50m`) and 0.607 / 0.617 (`m-100m`) test macro-F1 in Turkish / English, while erasing most of the coding ability.

In [ ]:
def load_mix(name):
    directory = sorted((WORKDIR / name).glob("step-*"))[-1]
    model = GLMLadderModel(ModelConfig.from_dict(base_states[results[name]["size"]]["model_config"]))
    load_model(model, str(directory / "model.safetensors"), strict=True)
    return model.to(device).eval()


test = {}
for size in BASES:
    model = load_base(size)
    test[f"SFT {size}"] = review_metrics(model, "test")
    show(f"SFT {size} (test)", test[f"SFT {size}"], baseline[size]["keep"])
    del model
    name = selected[size]
    model = load_mix(name)
    test[name] = review_metrics(model, "test")
    show(f"{name} (test)", test[name], results[name]["keep"])
    del model
    torch.cuda.empty_cache()

## 12. Chat samples

Does the selected model still talk? Greedy decoding, at most 120 tokens.

In [ ]:
from glm_ladder.chat import END_TOKEN, prompt_ids, special_id
from glm_ladder.classify import conversation
from glm_ladder.generate import generate

QUESTIONS = [
    [{"role": "user", "content": "Give me three tips for staying focused while studying."}],
    [{"role": "user", "content": "Sağlıklı beslenme için üç öneri ver."}],
    [{"role": "user", "content": "Write a Python function is_even(n) that returns True if n is even, otherwise False."}],
    conversation("Kargo çok hızlı geldi, ürün de tam beklediğim gibi.", "tr"),
    conversation("The food was cold and the waiter ignored us for twenty minutes.", "en"),
]
end_id = special_id(tokenizer, END_TOKEN)
samples = {}
for size in BASES:
    model = load_mix(selected[size])
    samples[size] = []
    for messages in QUESTIONS:
        ids = generate(model, prompt_ids(tokenizer, messages), max_new_tokens=120, temperature=0.0, stop_id=end_id)
        answer = tokenizer.decode([t for t in ids if t != end_id])
        samples[size].append({"user": messages[0]["content"], "assistant": answer})
        print(f"[{size}] USER: {messages[0]['content'][:90]}\n[{size}] ASSISTANT: {answer}\n")
    del model
    torch.cuda.empty_cache()

## 13. Upload

Receipts, baselines, test results and samples go to the runs repo under `finetune-mix/`. The two selected models go to `runs/<run_name>/` (weights and `trainer.json` only, no optimizer state), where RL will start from.

In [ ]:
summary = {"schema_version": "1.0", "kind": "finetune_mix_summary", "commit": commit, "gpu": GPU,
           "replay_fraction": REPLAY_FRACTION, "weights": weights, "limits": LIMITS, "learning_rates": list(LRS),
           "baseline": baseline, "selected": selected, "test": test, "samples": samples}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n")
api.upload_folder(folder_path=str(OUT), path_in_repo="finetune-mix", repo_id=RUNS_REPO, repo_type="model",
                  commit_message=f"joint fine-tuning receipts, commit {commit}")
for name in selected.values():
    directory = sorted((WORKDIR / name).glob("step-*"))[-1]
    (directory / "optimizer.pt").unlink(missing_ok=True)
    RunStore(api, RUNS_REPO, name).push(directory, metrics=WORKDIR / name / "metrics.jsonl", milestone=True)
    print("uploaded", name)

## 14. Done

Close the runtime. Locally:

```bash
python scripts/fetch_evals.py --pattern "finetune-mix/*.json" --to artifacts/finetune-mix
```